# # Lab 2 — Cleaning & Normalization Pipeline

In [2]:
import pandas as pd
import numpy as np
import json
import os
import sys

In [ ]:
if 'google.colab' in sys.modules: 
    if not os.path.exists('/content/nlp_course_project'):
        !git clone https://github.com/Karoshi-man/nlp_course_project.git
    
    %cd /content/nlp_course_project
    !pip install ftfy regex pandas -q
    sys.path.append('/content/nlp_course_project')
    
    FOLDER_ID = '1pIDpBFJ33L9XrldgXEXiAnLRNCs6f0gb'
    
    os.makedirs('/content/nlp_course_project/data', exist_ok=True)
    !gdown --folder https://drive.google.com/drive/folders/{FOLDER_ID} -O /content/nlp_course_project/data/
    
    data_dir = '/content/nlp_course_project/data'

else:
    sys.path.append(os.path.abspath('..'))
    data_dir = '../data'

In [4]:
raw_path = f'{data_dir}/raw/raw.csv'
out_dir = f'{data_dir}/processed_v2'

df = pd.read_csv(raw_path)
print(f"Кількість рядків: {len(df)}")

Кількість рядків: 583


In [5]:
text_column = 'description' if 'description' in df.columns else df.columns[0]

In [6]:
from src.preprocess import preprocess, PreprocessPolicy

policy = PreprocessPolicy(normalize_homoglyphs=False)

processed_results = df[text_column].astype(str).apply(lambda x: preprocess(x, policy))

df['clean_text'] = processed_results.apply(lambda d: d['clean_text'])
df['sentences'] = processed_results.apply(lambda d: d.get('sentences', []))

out_dir = f'{data_dir}/processed_v2'
os.makedirs(out_dir, exist_ok=True)
out_file = f'{out_dir}/processed_v2.csv'

df.to_csv(out_file, index=False)
print(f"успішно збережено у: {out_file}")


успішно збережено у: ../data/processed_v2/processed_v2.csv


In [7]:
empty_before = (df[text_column].astype(str).str.strip().str.len() < 10).sum()
empty_after = (df['clean_text'].str.strip().str.len() < 10).sum()
print(f"Порожніх/коротких рядків: {empty_before} -> {empty_after}")

dupes_before = round((df.duplicated(subset=[text_column]).sum() / len(df)) * 100, 2)
dupes_after = round((df.duplicated(subset=['clean_text']).sum() / len(df)) * 100, 2)
print(f"Відсоток дублікатів: {dupes_before}% -> {dupes_after}%")

len_before = df[text_column].astype(str).str.len().mean()
len_after = df['clean_text'].str.len().mean()
print(f"Середня довжина (символів): {int(len_before)} -> {int(len_after)}")

url_masked = df['clean_text'].str.contains('<URL>', regex=False).sum()
email_masked = df['clean_text'].str.contains('<EMAIL>', regex=False).sum()

print(f"Замасковано URL: у {url_masked} рядках")
print(f"Замасковано Email: у {email_masked} рядках")

display(df[[text_column, 'clean_text']].head(15))

Порожніх/коротких рядків: 0 -> 0
Відсоток дублікатів: 0.0% -> 0.0%
Середня довжина (символів): 3390 -> 3274
Замасковано URL: у 23 рядках
Замасковано Email: у 19 рядках


,description,clean_text
0,Warbirds Бойові Птахи України Всі вакансії ком...,Warbirds Бойові Птахи України Всі вакансії ком...
1,HOLYWATER TECH Всі вакансії компанії\r\nHOLYWA...,HOLYWATER TECH Всі вакансії компанії\nHOLYWATE...
2,SKELAR Всі вакансії компанії\r\nМи venture bui...,SKELAR Всі вакансії компанії\nМи venture build...
3,Spendbase Всі вакансії компанії\r\n“Startup wi...,"Spendbase Всі вакансії компанії\n""Startup with..."
4,SIXT Всі вакансії компанії\r\nSIXT is one of t...,SIXT Всі вакансії компанії\nSIXT is one of the...
5,AMO Всі вакансії компанії\r\nAMO — українська ...,AMO Всі вакансії компанії\nAMO - українська пр...
6,Ajax Systems Всі вакансії компанії\r\nAjax Sys...,Ajax Systems Всі вакансії компанії\nAjax Syste...
7,FREITTY Всі вакансії компанії\r\nFREITTY — це ...,FREITTY Всі вакансії компанії\nFREITTY - це ін...
8,Fuelfinance Всі вакансії компанії\r\nFuelfinan...,Fuelfinance Всі вакансії компанії\nFuelfinance...
9,OBRIO Всі вакансії компанії\r\nOBRIO is an IT ...,OBRIO Всі вакансії компанії\nOBRIO is an IT co...


In [8]:
edge_path = 'tests/edge_cases.jsonl'

edge_rows = [json.loads(line) for line in open(edge_path, encoding='utf-8') if line.strip()]

results = []
for r in edge_rows:
    raw = r['raw_text']
    out_1 = preprocess(raw, policy)
    clean_1 = out_1['clean_text']
    
    is_idempotent = (clean_1 == preprocess(clean_1, policy)['clean_text'])
    
    no_explosion = not (len(raw.strip()) > 0 and len(clean_1.strip()) == 0)
    
    results.append({
        'id': r['id'], 
        'raw': raw, 
        'clean': clean_1, 
        'idempotent': is_idempotent, 
        'no_explosion': no_explosion,
        'sentences': out_1.get('sentences', [])
    })

tests_df = pd.DataFrame(results)
print(f"Edge cases протестовано: {len(tests_df)}")

failed_tests = tests_df[~tests_df['idempotent'] | ~tests_df['no_explosion']]
if not failed_tests.empty:
    print("Є помилки в регресії:")
    display(failed_tests)
else:
    print("Усі тести пройшли успішно")

print("\n5 цікавих прикладів Edge Cases")
display(tests_df[['id', 'raw', 'clean', 'sentences']].head(5))

FileNotFoundError: [Errno 2] No such file or directory: 'tests/edge_cases.jsonl'